## ChromaDB 

In [1]:
### Building Langchain with ChromaDB

In [1]:
import os



In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma


#utility
import numpy as np
from typing import List,Dict,Any


C:\Users\kanha\AppData\Local\Temp\ipykernel_23780\3392625172.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# 1) Document Loading By creating sample data

In [3]:
## create sample data

sample_docs=[
     """
Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed.
The main types of machine learning are supervised learning, unsupervised
learning, and reinforcement learning. Common algorithms include Linear
Regression, Logistic Regression, Decision Trees, Random Forests, Support
Vector Machines, and K-Means clustering. Model evaluation metrics include
accuracy, precision, recall, F1-score, and ROC-AUC.
""",
"""
Deep Learning Fundamental

Deep Learning is a subset of machine learning that uses neural networks with
multiple hidden layers. A neural network consists of input, hidden, and output
layers connected by weighted edges. Popular architectures include CNNs for
computer vision, RNNs and LSTMs for sequential data, and Transformers for
natural language processing. Training uses backpropagation and gradient descent
to minimize a loss function.
""",
 """
Large Language Model Fundamentals

Large Language Models (LLMs) are transformer-based neural networks trained on
massive text datasets. They generate text by predicting the next token in a
sequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts
include tokenization, embeddings, context windows, prompting, fine-tuning,
Retrieval-Augmented Generation (RAG), and inference.
"""
]
sample_docs

['\nMachine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall, F1-score, and ROC-AUC.\n',
 '\nDeep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transformers for\nnatural language processing. Training uses backpropagation and gradient descent\nto minimize a loss function.\n',
 '\nL

In [4]:
##save sample
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)
print(f"Sample doc created at :-> {temp_dir}")

Sample doc created at :-> C:\Users\kanha\AppData\Local\Temp\tmpwsc019av


## 2) Document Loader

In [5]:
from langchain_community.document_loaders import DirectoryLoader
loader=DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)
documents=loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200]+"...")

Loaded 3 documents

First document preview:

Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed.
The main types of m...


## 3) Document Splitting

In [6]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "])
chunks=text_splitter.split_documents(documents=documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
print(f"\nChunk example")
print(f"Content {chunks[0].page_content[:150]}....\n")
print(f"Metadata {chunks[0].metadata}")

Created 4 chunks from 3 documents.

Chunk example
Content Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without....

Metadata {'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpwsc019av\\doc_0.txt'}


In [7]:
chunks

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpwsc019av\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall,'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpwsc019av\\doc_0.txt'}, page_content='metrics include\naccuracy, precision, recall, F1-score, and ROC-AUC.'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpwsc019av\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses

### 4) Embedding Models


In [8]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## 5) ChromaDB Time

In [9]:
## create vector store
persist_directory="./chroma_db"

##Initialisation chromadb with hugging face
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(
          model_name="sentence-transformers/all-MiniLM-L6-v2"
    ),
    persist_directory=persist_directory,
    collection_name="rag_collection",
   

)
print(f" Vector store created at {persist_directory} with {vectorstore._collection.count()} vectors")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Vector store created at ./chroma_db with 16 vectors


## Test similiarity Check

In [10]:
query="What are the types of machine learning?"
similiar_docs=vectorstore.similarity_search(query,k=4)
similiar_docs

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall,'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning,

In [11]:
query="What is NLP"
similiar_docs=vectorstore.similarity_search(query,k=4)
similiar_docs

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generati

## Advanced similiaruty search

In [12]:
results=vectorstore.similarity_search_with_score(query,k=4)
results

[(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
  1.446568250656128),
 (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetr

In [13]:
results=vectorstore.similarity_search_with_relevance_scores(query,k=4)
results

C:\Users\kanha\AppData\Local\Temp\ipykernel_23780\3467109865.py:1: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'), -0.02287821948810942), (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include G

[(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
  -0.02287821948810942),
 (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nR

### Initialize LLM ,RAG Chain,Promp Template,Query The RAG System

In [14]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

In [15]:
llm_pred=llm.invoke("What is LLM")
print(llm_pred)

content="LLM stands for Large Language Model. It's a type of artificial intelligence (AI) designed to process and understand human language at a massive scale.\n\nA Large Language Model is a deep learning-based neural network that uses complex algorithms to learn patterns and relationships in large datasets of text. The model is trained on vast amounts of text data, allowing it to improve its language understanding and generation capabilities over time.\n\nLLMs are typically trained on unlabeled text data, such as books, articles, and websites, which enables them to learn the structure and semantics of language. This training process allows LLMs to generate human-like responses to a wide range of questions and tasks, making them useful for applications like:\n\n1. **Language translation**: LLMs can translate languages in real-time, with high accuracy.\n2. **Text summarization**: They can summarize long documents into concise summaries.\n3. **Chatbots and conversational AI**: LLMs power

## Modern RAG Chain

In [16]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [17]:
### Convert vector store to retriever

retriever=vectorstore.as_retriever(
    search_kwarg={"k":3}

)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000018D46334890>, search_kwargs={})

In [18]:
## Create prompt template
system_prompt="""You are an asisstant for question-answering tasks.
use the follwing pieceof retrieved context to answer the question
If you don't know the answer,just sat that you don't know.
Use three senetenes maximum and keep the answer concise.
Context:{context}"""
prompt=ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [19]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [20]:
### Create a document chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2')
| StrOutputPars

In [22]:
### Create final rag chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000018D46334890>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't

In [24]:
response=rag_chain.invoke({
    "input":"What is deep Learning"
})

In [25]:
response

{'input': 'What is deep Learning',
 'context': [Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transformers for\nnatural language processing. Training uses backpropagation and gradient descent\nto minimize a loss function.'),
  Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision,

In [26]:
response["answer"]

'Deep learning is a subset of machine learning that uses neural networks with multiple hidden layers to analyze and interpret data.'